# FP-Cox Optimizer — SaDE v7

Self-Adaptive Differential Evolution for Fractional Polynomial Cox model selection.

**New in v7 — Four-model comparison:**

| Model | Type | Covariates | PH required? | C-index | BIC | CV |
|---|---|---|---|---|---|---|
| **Kaplan-Meier** | Non-parametric | None (marginal) | No | — | — | — |
| **Cox PH (traditional)** | Semi-parametric | Linear | Yes | ✓ | ✓ | ✓ |
| **Weibull AFT** | Fully parametric | Linear | No | ✓ | ✓ | ✓ |
| **FP Cox (SaDE v7)** | Semi-parametric | FP-transformed | Yes | ✓ | ✓ | ✓ |

**All v6 features retained:**
- Schoenfeld residual PH test (Cox PH and FP Cox only)
- Model equations printer (Cox PH, Weibull AFT, FP Cox)
- 4-model cross-validation with pairwise comparison
- scikit-learn-compatible CV wrapper
- Algorithm validation (multi-seed stability)

In [1]:
"""
FP-Cox Optimizer — SaDE v7
All fixes from v5 and additions from v6 are retained.

New in v7
---------
ADD 6  (Methodological)
    Kaplan-Meier baseline added to _fit_final_models().
    KM is the marginal (covariate-free) non-parametric estimator.
    It serves as a lower-bound reference for IBS (any adjusted model
    should beat the KM IBS).  C-index and BIC are not applicable.
    If strata_cols are present, a stratified KM is also fitted.

ADD 7  (Methodological)
    Weibull AFT model added to _fit_final_models().
    WeibullAFTFitter from lifelines uses the SAME linear covariates as
    the traditional Cox PH, allowing a direct parametric vs
    semi-parametric comparison.  Equation:
        log T = β₀ + Σ βⱼ xⱼ + σ ε,  ε ~ Gumbel(0,1)
    BIC uses the full (not partial) log-likelihood and total n
    (censored + event), because AFT conditions on all observations.

ADD 8  (Visualisation)
    _plot_survival_curves() — overlaid KM, Cox PH, Weibull AFT, and
    FP Cox survival curves for an average-covariate profile, allowing
    visual comparison of the four estimated survival functions.

ADD 9  (Validation)
    run_cross_validation() updated to 3-model CV (Cox PH, Weibull AFT,
    FP Cox) with an all-pairs comparison table.  KM is excluded from
    CV because it has no covariates and cannot produce per-fold
    concordance indices.

ADD 10 (Interpretability)
    _print_model_equations() updated to include the Weibull AFT
    log-time equation alongside the Cox PH and FP Cox log-HR equations.
"""

'\nFP-Cox Optimizer — SaDE v7\nAll fixes from v5 and additions from v6 are retained.\n\nNew in v7\n---------\nADD 6  (Methodological)\n    Kaplan-Meier baseline added to _fit_final_models().\n    KM is the marginal (covariate-free) non-parametric estimator.\n    It serves as a lower-bound reference for IBS (any adjusted model\n    should beat the KM IBS).  C-index and BIC are not applicable.\n    If strata_cols are present, a stratified KM is also fitted.\n\nADD 7  (Methodological)\n    Weibull AFT model added to _fit_final_models().\n    WeibullAFTFitter from lifelines uses the SAME linear covariates as\n    the traditional Cox PH, allowing a direct parametric vs\n    semi-parametric comparison.  Equation:\n        log T = β₀ + Σ βⱼ xⱼ + σ ε,  ε ~ Gumbel(0,1)\n    BIC uses the full (not partial) log-likelihood and total n\n    (censored + event), because AFT conditions on all observations.\n\nADD 8  (Visualisation)\n    _plot_survival_curves() — overlaid KM, Cox PH, Weibull AFT, and\n

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from lifelines import CoxPHFitter, WeibullAFTFitter, KaplanMeierFitter
from lifelines.utils import k_fold_cross_validation
from lifelines.statistics import multivariate_logrank_test
import warnings
from collections import deque
from dataclasses import dataclass, field
from typing import List, Optional, Union
from scipy import stats as scipy_stats
from itertools import combinations

warnings.filterwarnings('ignore')

try:
    from sksurv.metrics import integrated_brier_score
    from sksurv.util import Surv
    HAS_SKSURV = True
except ImportError:
    HAS_SKSURV = False
    print('scikit-survival not found — IBS skipped.')

In [3]:
# Result container
@dataclass
class _SaDEResult:
    x:       np.ndarray
    fun:     float
    nfev:    int
    ngen:    int
    history: List[float] = field(default_factory=list)

In [4]:
# ---------------------------------------------------------------------------
# _choose: accept int OR list for excl
# ---------------------------------------------------------------------------

def _choose(n: int, k: int, excl: Union[int, list], rng) -> np.ndarray:
    mask = np.ones(n, dtype=bool)
    if isinstance(excl, int):
        excl = [excl]
    for e in excl:
        mask[e] = False
    pool = np.where(mask)[0]
    if len(pool) < k:
        return rng.choice(pool, size=k, replace=True)
    return rng.choice(pool, size=k, replace=False)

In [5]:
# Mutation strategies

def _s1(pop, F, t, b, rng):   # DE/rand/1
    r1, r2, r3 = _choose(len(pop), 3, t, rng)
    return pop[r1] + F * (pop[r2] - pop[r3])

def _s2(pop, F, t, b, rng):   # DE/current-to-best/2
    r1, r2, r3, r4 = _choose(len(pop), 4, [t, b], rng)
    return (pop[t] + F*(pop[b]-pop[t]) + F*(pop[r1]-pop[r2]) + F*(pop[r3]-pop[r4]))

def _s3(pop, F, t, b, rng):   # DE/current-to-rand/1
    r1, r2, r3 = _choose(len(pop), 3, t, rng)
    return pop[t] + F*(pop[r1]-pop[t]) + F*(pop[r2]-pop[r3])

def _s4(pop, F, t, b, rng):   # DE/rand/2
    r1, r2, r3, r4, r5 = _choose(len(pop), 5, t, rng)
    return pop[r1] + F*(pop[r2]-pop[r3]) + F*(pop[r4]-pop[r5])

_STRATS = [_s1, _s2, _s3, _s4]
_NS     = len(_STRATS)

In [6]:
# ---------------------------------------------------------------------------
# Boundary repair and crossover
# ---------------------------------------------------------------------------

def _repair(v, lb, ub):
    v = np.round(v).astype(int)
    lo = v < lb;  v[lo] = lb[lo] + (lb[lo] - v[lo])
    hi = v > ub;  v[hi] = ub[hi] - (v[hi] - ub[hi])
    return np.clip(v, lb, ub)

def _cross(x, v, CR, rng):
    d = len(x)
    m = rng.random(d) < CR
    m[rng.integers(d)] = True
    return np.where(m, v, x)

In [7]:
# SaDE engine

class _SaDE:
    """
    Self-Adaptive Differential Evolution for integer search spaces.
    Strategy pool: DE/rand/1, DE/current-to-best/2,
                   DE/current-to-rand/1, DE/rand/2.
    F ~ Cauchy(0.5, 0.3)  [JADE-style extension].
    CR adapted via mean of successful CRs  [original SaDE spec].
    """

    def __init__(self, func, bounds, pop_size=50, max_evals=1000,
                 lp=10, patience=10, seed=None, callback=None):
        self.func     = func
        self.dim      = len(bounds)
        self.lb       = np.array([b[0] for b in bounds], dtype=int)
        self.ub       = np.array([b[1] for b in bounds], dtype=int)
        self.N        = pop_size
        self.maxev    = max_evals
        self.lp       = lp
        self.patience = patience
        self.cb       = callback
        self.rng      = np.random.default_rng(seed)
        self.p        = np.ones(_NS) / _NS
        self.crm      = np.full(_NS, 0.5)
        self.ns       = [deque() for _ in range(_NS)]
        self.nf       = [deque() for _ in range(_NS)]
        self.crok     = [deque() for _ in range(_NS)]
        self.strategy_counts  = np.zeros(_NS, dtype=int)
        self.strategy_success = np.zeros(_NS, dtype=int)

    def run_opt(self, init_pop=None):
        if init_pop is not None and init_pop.shape == (self.N, self.dim):
            pop = np.clip(np.round(init_pop).astype(int), self.lb, self.ub)
        else:
            pop = self.rng.integers(self.lb, self.ub + 1, size=(self.N, self.dim))

        fit        = np.array([self.func(pop[i]) for i in range(self.N)])
        nfev       = self.N
        bi         = int(np.argmin(fit))
        hist       = [float(fit[bi])]
        gen        = 0
        no_improve = 0
        best_ever  = float(fit[bi])

        while nfev < self.maxev:
            gen += 1
            for i in range(self.N):
                if nfev >= self.maxev:
                    break
                k  = self.rng.choice(_NS, p=self.p)
                F  = float(np.clip(self.rng.standard_cauchy()*0.3+0.5, 1e-6, 2.0))
                CR = float(np.clip(self.rng.normal(self.crm[k], 0.1), 0.0, 1.0))
                v  = _STRATS[k](pop, F, i, bi, self.rng).astype(float)
                u  = _repair(_cross(pop[i].astype(float), v, CR, self.rng),
                             self.lb, self.ub)
                fu = self.func(u);  nfev += 1
                self.strategy_counts[k] += 1
                if fu <= fit[i]:
                    pop[i], fit[i] = u, fu
                    self.ns[k].append(1)
                    self.crok[k].append(CR)
                    self.strategy_success[k] += 1
                    if fu < fit[bi]: bi = i
                else:
                    self.nf[k].append(1)

            if gen % self.lp == 0:
                self._upd_p()
                self._upd_crm()

            hist.append(float(fit[bi]))
            if self.cb: self.cb(pop[bi], fit[bi], gen)

            if self.patience > 0:
                if fit[bi] < best_ever - 1e-8:
                    best_ever = float(fit[bi]);  no_improve = 0
                else:
                    no_improve += 1
                if no_improve >= self.patience:
                    print(f'  Early stop at gen {gen} '
                          f'(no improvement for {self.patience} gens, evals: {nfev})')
                    break

        return _SaDEResult(x=pop[bi].copy(), fun=float(fit[bi]),
                           nfev=nfev, ngen=gen, history=hist)

    def _upd_p(self):
        ns = np.array([sum(q) for q in self.ns], dtype=float)
        nf = np.array([sum(q) for q in self.nf], dtype=float)
        with np.errstate(divide='ignore', invalid='ignore'):
            r = np.where(ns+nf > 0, ns/(ns+nf), 0.0)
        tot = r.sum()
        if tot > 0:
            self.p = 0.05 + 0.95*r/tot
            self.p /= self.p.sum()
        for k in range(_NS):
            while len(self.ns[k]) > self.lp: self.ns[k].popleft()
            while len(self.nf[k]) > self.lp: self.nf[k].popleft()

    def _upd_crm(self):
        for k in range(_NS):
            if self.crok[k]:
                self.crm[k] = float(np.mean(list(self.crok[k])))
            while len(self.crok[k]) > self.lp: self.crok[k].popleft()


print('SaDE v7 engine loaded.')

SaDE v7 engine loaded.


In [8]:
# FPCoxOptimizer v7
class FPCoxOptimizer:
    """
    FP-Cox Optimizer v7 — Four-model comparison.

    Models fitted and compared
    --------------------------
    1. Kaplan-Meier (KM)
       Non-parametric, marginal survival curve.  No covariates, no PH
       assumption.  Serves as the IBS baseline — any covariate model
       should achieve lower IBS than KM.
       Metrics : IBS only  (C-index = N/A, BIC = N/A)

    2. Traditional Cox PH
       Semi-parametric, linear covariates, proportional hazards assumed.
       log[ h(t|x)/h₀(t) ] = Σ βⱼ xⱼ
       BIC uses n_events and unpenalized partial log-likelihood.
       Metrics : C-index, BIC, IBS, CV

    3. Weibull AFT
       Fully parametric, linear covariates, no PH assumption required.
       log T = μ₀ + Σ βⱼ xⱼ + σ ε,   ε ~ Gumbel(0,1)
       BIC uses total n (events + censored) and full log-likelihood.
       Metrics : C-index, AIC, BIC, IBS, CV

    4. FP Cox (SaDE v7)
       Semi-parametric, FP2-transformed covariates, PH assumed.
       log[ h(t|x)/h₀(t) ] = Σ βⱼ φⱼ(x)  (φ = FP terms)
       Powers selected by SaDE minimising BIC.
       Metrics : C-index, BIC, IBS, CV

    FP1 vs FP2
    ----------
    Each covariate has TWO power slots.  p2=None → FP1; otherwise FP2.
    The algorithm selects FP1 or FP2 adaptively per covariate.

    Simultaneous optimisation
    -------------------------
    All covariates are optimised at once.  dim = 2 × len(covariates).
    """

    POWER_SET = [None, -3, -2.5, -2, -1.5, -1, -0.5, -0.25,
                 0,    0.25, 0.5,  1,  1.5,  2,  2.5,  3]
    N_POWERS  = len(POWER_SET)  # 16

    def __init__(self, df, covariates, duration_col, event_col,
                 penalizer=0.01, gamma=0.0, strata_cols=None, df_test=None):

        self.covariates   = covariates
        self.duration_col = duration_col
        self.event_col    = event_col
        self.penalizer    = penalizer
        self.gamma        = gamma
        self.strata_cols  = strata_cols or []

        self.df, self._scales = self._preprocess_positive(df, covariates)

        if df_test is not None:
            self.df_test, _ = self._preprocess_positive(
                df_test, covariates, scales=self._scales)
        else:
            self.df_test = None

        self._n_events = int(self.df[self.event_col].sum())
        self._n_total  = len(self.df)

        # Precompute all FP transforms
        self._precomp = {}
        self._log_x   = {}
        for col in covariates:
            x     = self.df[col].values.astype(float)
            log_x = np.log(x)
            self._log_x[col] = log_x
            for p in [p for p in self.POWER_SET if p is not None]:
                z = log_x if p == 0 else np.power(x, p)
                if np.isfinite(z).all():
                    self._precomp[(col, p)] = z

        const_cols = {
            self.duration_col: self.df[self.duration_col].values,
            self.event_col:    self.df[self.event_col].values,
        }
        for c in self.strata_cols:
            const_cols[c] = self.df[c].values
        self._const_df = pd.DataFrame(const_cols, index=self.df.index)

        self.evaluation_cache  = {}
        self.best_val          = np.inf
        self.history: list     = []
        self.best_powers: list = []

        # Model objects — all four
        self.km_model          = None   # KaplanMeierFitter
        self.km_strat_models   = {}     # {stratum_val: KaplanMeierFitter}
        self.traditional_model = None   # CoxPHFitter (penalized)
        self.weibull_aft_model = None   # WeibullAFTFitter
        self.final_fp_model    = None   # CoxPHFitter (FP, penalized)

        # DataFrames
        self._df_trad_final    = None
        self._df_fp_final      = None

        self.metrics_: dict    = {}
        self.cv_results_: dict = {}

    # -----------------------------------------------------------------------
    # Preprocessing
    # -----------------------------------------------------------------------

    @staticmethod
    def _preprocess_positive(df, features, scales=None):
        df = df.copy()
        computed_scales = {}
        for col in features:
            x = df[col].astype(float)
            if (x <= 0).any():
                x = x - x.min() + 1e-5
            if scales is not None:
                scale = scales[col]
            else:
                mean_abs = np.mean(np.abs(x))
                scale = 1.0 if mean_abs == 0 else 10.0**np.floor(np.log10(mean_abs))
            computed_scales[col] = scale
            df[col] = x / scale
        return df, computed_scales

    @staticmethod
    def _canonical_key(indices):
        key = list(indices)
        for i in range(0, len(key), 2):
            a, b = key[i], key[i+1]
            if b == 0 and a != 0:  key[i], key[i+1] = 0, a
            elif a != 0 and b != 0: key[i], key[i+1] = min(a,b), max(a,b)
        return tuple(key)

    def _generate_fp_features(self, features, powers):
        transformed = {}
        for col, (p1, p2) in zip(features, powers):
            active = sorted([p for p in (p1, p2) if p is not None])
            if not active: continue
            if len(active) == 1:
                p = active[0]
                arr = self._precomp.get((col, p))
                if arr is None: return None
                transformed[f'{col}_fp_{p}'] = arr
            else:
                pa, pb = active
                za = self._precomp.get((col, pa))
                if za is None: return None
                transformed[f'{col}_fp1_{pa}'] = za
                if pa == pb:
                    transformed[f'{col}_fp2_rep_{pb}'] = za * self._log_x[col]
                else:
                    zb = self._precomp.get((col, pb))
                    if zb is None: return None
                    transformed[f'{col}_fp2_{pb}'] = zb
        return transformed

    # Objective function (BIC-based, Cox only)

    def _objective_function(self, x):
        key = self._canonical_key(x)
        if key in self.evaluation_cache:
            return self.evaluation_cache[key]

        powers = [
            (self.POWER_SET[key[2*i]], self.POWER_SET[key[2*i+1]])
            for i in range(len(self.covariates))
        ]
        fp_cols = self._generate_fp_features(self.covariates, powers)
        if not fp_cols:
            self.evaluation_cache[key] = 1e10
            return 1e10

        df_model = self._const_df.copy()
        for col_name, arr in fp_cols.items():
            df_model[col_name] = arr

        strata = self.strata_cols or None
        try:
            cph_unpen = CoxPHFitter(penalizer=0.0)
            cph_unpen.fit(df_model, duration_col=self.duration_col,
                          event_col=self.event_col, strata=strata,
                          show_progress=False)
            k   = len(cph_unpen.params_)
            bic = -2*cph_unpen.log_likelihood_ + k*np.log(self._n_events)
            ci  = 0.0
            if self.gamma > 0:
                cph_pen = CoxPHFitter(penalizer=self.penalizer)
                cph_pen.fit(df_model, duration_col=self.duration_col,
                            event_col=self.event_col, strata=strata,
                            show_progress=False)
                ci = cph_pen.concordance_index_
            val = bic - self.gamma*self._n_events*ci
        except Exception:
            val = 1e10

        self.evaluation_cache[key] = val
        if val < self.best_val: self.best_val = val
        return val

    def _warm_start_pop(self, pop_size, rng):
        dim  = 2*len(self.covariates)
        pop  = rng.integers(0, self.N_POWERS, size=(pop_size, dim))
        NA, LIN, SQRT, LG, Q = 0, 11, 10, 8, 13
        seeds = [
            np.tile([LIN,  NA],   len(self.covariates)),
            np.tile([SQRT, NA],   len(self.covariates)),
            np.tile([LG,   NA],   len(self.covariates)),
            np.tile([LIN,  SQRT], len(self.covariates)),
            np.tile([LIN,  Q],    len(self.covariates)),
        ]
        for row, s in enumerate(seeds[:min(len(seeds), pop_size)]):
            pop[row] = s
        return pop

    def _callback(self, best_x, best_f, gen):
        self.history.append(self.best_val)

    # IBS helper

    def _compute_ibs(self, cph, df_for_model, df_test_model=None, y_test=None):
        if not HAS_SKSURV: return None
        try:
            y_train = Surv.from_arrays(
                event=self.df[self.event_col].astype(bool).values,
                time =self.df[self.duration_col].values)
            t_min  = self.df[self.duration_col].min()
            t_max  = self.df[self.duration_col].max()
            times  = np.linspace(t_min, t_max*0.999, 100)
            if df_test_model is not None and y_test is not None:
                surv = cph.predict_survival_function(df_test_model, times=times)
                return float(integrated_brier_score(y_train, y_test, surv.T.values, times))
            else:
                print('  WARNING: IBS on training data (optimistic).')
                surv = cph.predict_survival_function(df_for_model, times=times)
                return float(integrated_brier_score(y_train, y_train, surv.T.values, times))
        except Exception: return None

    def _compute_ibs_km(self):
        """
        IBS for KM: all subjects receive the same marginal survival curve.
        This is the covariate-free baseline — any adjusted model should
        achieve lower IBS.
        """
        if not HAS_SKSURV or self.km_model is None: return None
        try:
            y = Surv.from_arrays(
                event=self.df[self.event_col].astype(bool).values,
                time =self.df[self.duration_col].values)
            t_min  = self.df[self.duration_col].min()
            t_max  = self.df[self.duration_col].max()
            times  = np.linspace(t_min, t_max*0.999, 100)
            km_sf  = self.km_model.survival_function_at_times(times).values
            # Broadcast: every subject gets the same KM curve
            n      = len(self.df)
            surv_matrix = np.tile(km_sf, (n, 1))  # (n, len(times))
            return float(integrated_brier_score(y, y, surv_matrix, times))
        except Exception as e:
            print(f'  [!] KM IBS failed: {e}')
            return None

    # -----------------------------------------------------------------------
    # Main optimize entry point
    # -----------------------------------------------------------------------

    def optimize(self, maxiter=15, popsize=8, seed=42, max_evals=None):
        dim      = 2*len(self.covariates)
        pop_size = popsize*dim
        if max_evals is None:
            max_evals = pop_size*maxiter
        lp       = max(3, maxiter//5)
        patience = max(3, maxiter//3)

        print(f'Starting SaDE v7  (gamma={self.gamma})')
        print(f'  FP covariates : {self.covariates}')
        print(f'  Simultaneous  : {len(self.covariates)} covariates '
              f'| dim={dim}  (FP2 max; FP1 if p2=None)')
        if self.strata_cols:
            print(f'  Strata        : {self.strata_cols}')
        print(f'  Scales        : '
              + ', '.join(f'{c}÷{s:.3g}' for c,s in self._scales.items()))
        print(f'  n={self._n_total}, n_events={self._n_events}')
        print(f'  Pop={pop_size} | MaxGens={maxiter} | Budget={max_evals}')

        rng      = np.random.default_rng(seed)
        init_pop = self._warm_start_pop(pop_size, rng)
        engine   = _SaDE(
            func=self._objective_function,
            bounds=[(0, self.N_POWERS-1)]*dim,
            pop_size=pop_size, max_evals=max_evals,
            lp=lp, patience=patience, seed=seed, callback=self._callback)
        result = engine.run_opt(init_pop=init_pop)

        best_indices = result.x
        print('\n--- Optimal Power Selection ---')
        for i in range(len(self.covariates)):
            p1 = self.POWER_SET[best_indices[2*i]]
            p2 = self.POWER_SET[best_indices[2*i+1]]
            self.best_powers.append((p1, p2))
            fp_type = 'FP1' if p2 is None else 'FP2'
            print(f'  {self.covariates[i]:<22}: p1={str(p1):<7} p2={str(p2):<7}  [{fp_type}]')
        print(f'\n  Best BIC  : {result.fun:.4f}')
        print(f'  Gens      : {result.ngen}')
        print(f'  Evals     : {result.nfev} '
              f'(cache hits: {result.nfev - len(self.evaluation_cache)})')

        # Strategy stats
        names = ['DE/rand/1','DE/curr-to-best/2','DE/curr-to-rand/1','DE/rand/2']
        print('\n--- Strategy Usage ---')
        for i, name in enumerate(names):
            use = engine.strategy_counts[i]
            suc = engine.strategy_success[i]
            rate = 100*suc/use if use > 0 else 0.0
            print(f'  {name:<28}: used={use:4d}  success={suc:4d}  rate={rate:5.1f}%')

        self._plot_convergence()
        self._fit_final_models()

    def _plot_convergence(self):
        plt.figure(figsize=(10, 4))
        plt.plot(range(1, len(self.history)+1), self.history,
                 marker='o', ms=3, color='steelblue')
        plt.title('SaDE v7 Convergence')
        plt.xlabel('Generation')
        plt.ylabel('BIC (unpenalized)')
        plt.grid(True, alpha=0.4)
        plt.tight_layout()
        plt.show()

    # ADD 6 + ADD 7: Fit all four models

    def _fit_final_models(self):
        print('\n' + '='*72)
        print('FOUR-MODEL COMPARISON')
        print('='*72)

        strata = self.strata_cols or None

        # --- Build traditional covariate DataFrame ---
        seen, trad_cols = set(), []
        for c in (self.covariates + self.strata_cols +
                  [self.duration_col, self.event_col]):
            if c not in seen:
                trad_cols.append(c)
                seen.add(c)
        self._df_trad_final = self.df[trad_cols].copy()

        # ── 1. Kaplan-Meier (ADD 6) ──────────────────────────────────────────
        print('\n[1/4] Fitting Kaplan-Meier...')
        self.km_model = KaplanMeierFitter()
        self.km_model.fit(
            durations  = self.df[self.duration_col],
            event_observed = self.df[self.event_col],
            label      = 'Kaplan-Meier (marginal)',
        )
        # Stratified KM (if strata_cols available, use first stratum col)
        self.km_strat_models = {}
        if self.strata_cols:
            strat_col = self.strata_cols[0]
            for val in sorted(self.df[strat_col].unique()):
                mask = self.df[strat_col] == val
                kmf  = KaplanMeierFitter()
                kmf.fit(
                    durations      = self.df.loc[mask, self.duration_col],
                    event_observed = self.df.loc[mask, self.event_col],
                    label          = f'KM {strat_col}={val}',
                )
                self.km_strat_models[val] = kmf

        ibs_km = self._compute_ibs_km()
        median_km = self.km_model.median_survival_time_
        print(f'   Median survival time : {median_km:.4f}')
        print(f'   IBS (train)          : {ibs_km:.4f}' if ibs_km else '   IBS : N/A')

        # ── 2. Traditional Cox PH ────────────────────────────────────────────
        print('\n[2/4] Fitting Traditional Cox PH...')
        self.traditional_model = CoxPHFitter(penalizer=self.penalizer)
        self.traditional_model.fit(
            self._df_trad_final,
            duration_col=self.duration_col, event_col=self.event_col,
            strata=strata, show_progress=False)

        _trad_unpen = CoxPHFitter(penalizer=0.0)
        _trad_unpen.fit(
            self._df_trad_final,
            duration_col=self.duration_col, event_col=self.event_col,
            strata=strata, show_progress=False)

        k_t   = len(_trad_unpen.params_)
        bic_t = -2*_trad_unpen.log_likelihood_ + k_t*np.log(self._n_events)
        ci_t  = self.traditional_model.concordance_index_
        ibs_t = self._compute_ibs(self.traditional_model, self._df_trad_final)
        print(f'   C-index : {ci_t:.4f}   BIC : {bic_t:.2f}'
              + (f'   IBS : {ibs_t:.4f}' if ibs_t else ''))

        # ── 3. Weibull AFT (ADD 7) ────────────────────────────────────────────
        print('\n[3/4] Fitting Weibull AFT...')
        #
        # WeibullAFTFitter notes
        # ----------------------
        # • Uses the FULL likelihood (events + censored) → BIC uses n_total.
        # • No strata argument: stratification is handled by adding stratum
        #   dummies. We include strata_cols as covariates for a fair comparison.
        # • The model has TWO sub-models:
        #     lambda_ (scale / AFT location): log T = μ₀ + Σ βⱼ xⱼ
        #     rho_    (shape): treated as constant by default
        # • lifelines provides AIC_ directly; BIC is computed manually.
        #
        aft_cols = [c for c in trad_cols if c != self.duration_col]
        df_aft   = self.df[trad_cols].copy()

        self.weibull_aft_model = WeibullAFTFitter(penalizer=self.penalizer)
        try:
            self.weibull_aft_model.fit(
                df_aft,
                duration_col   = self.duration_col,
                event_col      = self.event_col,
                show_progress  = False,
            )
            k_w   = self.weibull_aft_model.params_.shape[0]
            ll_w  = self.weibull_aft_model.log_likelihood_
            aic_w = self.weibull_aft_model.AIC_
            bic_w = -2*ll_w + k_w*np.log(self._n_total)   # full-likelihood BIC
            ci_w  = self.weibull_aft_model.concordance_index_
            ibs_w = self._compute_ibs(self.weibull_aft_model, df_aft)
            print(f'   C-index : {ci_w:.4f}   AIC : {aic_w:.2f}   BIC : {bic_w:.2f}'
                  + (f'   IBS : {ibs_w:.4f}' if ibs_w else ''))
        except Exception as e:
            print(f'   [!] Weibull AFT failed: {e}')
            self.weibull_aft_model = None
            k_w = ci_w = aic_w = bic_w = ibs_w = None

        # ── 4. FP Cox ─────────────────────────────────────────────────────────
        print('\n[4/4] Fitting FP Cox (SaDE v7)...')
        fp_cols = self._generate_fp_features(self.covariates, self.best_powers)
        if fp_cols is None: fp_cols = {}
        self._df_fp_final = self._const_df.copy()
        for col_name, arr in fp_cols.items():
            self._df_fp_final[col_name] = arr

        self.final_fp_model = CoxPHFitter(penalizer=self.penalizer)
        self.final_fp_model.fit(
            self._df_fp_final,
            duration_col=self.duration_col, event_col=self.event_col,
            strata=strata, show_progress=False)

        _fp_unpen = CoxPHFitter(penalizer=0.0)
        _fp_unpen.fit(
            self._df_fp_final,
            duration_col=self.duration_col, event_col=self.event_col,
            strata=strata, show_progress=False)

        k_fp   = len(_fp_unpen.params_)
        bic_fp = -2*_fp_unpen.log_likelihood_ + k_fp*np.log(self._n_events)
        ci_fp  = self.final_fp_model.concordance_index_
        ibs_fp = self._compute_ibs(self.final_fp_model, self._df_fp_final)
        print(f'   C-index : {ci_fp:.4f}   BIC : {bic_fp:.2f}'
              + (f'   IBS : {ibs_fp:.4f}' if ibs_fp else ''))

        # ── Metrics dict ──────────────────────────────────────────────────────
        self.metrics_ = {
            'Kaplan-Meier': {
                'C-index': 'N/A',
                'BIC'    : 'N/A',
                'AIC'    : 'N/A',
                'IBS'    : round(ibs_km,  4) if ibs_km  is not None else 'N/A',
                'k'      : 'N/A',
                'median_T': round(median_km, 4),
            },
            'Cox PH (trad)': {
                'C-index': round(ci_t,  4),
                'BIC'    : round(bic_t, 2),
                'AIC'    : 'N/A (partial)',
                'IBS'    : round(ibs_t,  4) if ibs_t  is not None else 'N/A',
                'k'      : k_t,
            },
            'Weibull AFT': {
                'C-index': round(ci_w,  4) if ci_w  is not None else 'N/A',
                'BIC'    : round(bic_w, 2) if bic_w is not None else 'N/A',
                'AIC'    : round(aic_w, 2) if aic_w is not None else 'N/A',
                'IBS'    : round(ibs_w,  4) if ibs_w  is not None else 'N/A',
                'k'      : k_w,
            },
            'FP Cox (SaDE v7)': {
                'C-index': round(ci_fp,  4),
                'BIC'    : round(bic_fp, 2),
                'AIC'    : 'N/A (partial)',
                'IBS'    : round(ibs_fp,  4) if ibs_fp  is not None else 'N/A',
                'k'      : k_fp,
            },
        }

        # ── Print comparison table ────────────────────────────────────────────
        self._print_comparison_table()

        # ── Equations (ADD 10) ────────────────────────────────────────────────
        self._print_model_equations(_trad_unpen, _fp_unpen)

        # ── Survival curve plot (ADD 8) ───────────────────────────────────────
        self._plot_survival_curves(df_aft)

        # ── PH tests (ADD 1 from v6 — Cox models only) ────────────────────────
        self._test_ph_assumption(
            self.traditional_model, self._df_trad_final, 'Traditional Cox PH')
        self._test_ph_assumption(
            self.final_fp_model, self._df_fp_final, 'FP Cox (SaDE v7)')

    # -----------------------------------------------------------------------
    # Comparison table
    # -----------------------------------------------------------------------

    def _print_comparison_table(self):
        bar  = '='*72
        sep  = '-'*72
        W    = 18
        models = list(self.metrics_.keys())
        print(f'\n{bar}')
        print('FOUR-MODEL SUMMARY TABLE')
        print(bar)

        # Header
        hdr = f"{'Metric':<16}"
        for m in models:
            hdr += f' | {m:>{W}}'
        print(hdr)
        print(sep)

        # Rows
        for metric in ['C-index', 'BIC', 'AIC', 'IBS', 'k']:
            row = f'{metric:<16}'
            for m in models:
                val = self.metrics_[m].get(metric, 'N/A')
                row += f' | {str(val):>{W}}'
            print(row)

        # Best-in-row markers
        print(sep)
        print('Best (↑ C-index, ↓ BIC, ↓ IBS):')
        for metric, higher_better in [('C-index', True), ('BIC', False), ('IBS', False)]:
            vals = {}
            for m in models:
                v = self.metrics_[m].get(metric, 'N/A')
                try: vals[m] = float(v)
                except: pass
            if vals:
                best = max(vals, key=vals.get) if higher_better \
                       else min(vals, key=vals.get)
                direction = '↑' if higher_better else '↓'
                print(f'  {metric:<10}: {best}  ({direction} {vals[best]:.4f})')

        # BIC improvement of FP over traditional Cox
        try:
            bic_t  = float(self.metrics_['Cox PH (trad)']['BIC'])
            bic_fp = float(self.metrics_['FP Cox (SaDE v7)']['BIC'])
            delta  = bic_t - bic_fp
            print(f'\n  ΔBIC (Cox PH − FP Cox) = {delta:.2f}  '
                  f'({"FP Cox" if delta > 0 else "Traditional Cox"} preferred by BIC)')
            if delta > 10:
                print('   → ΔBIC > 10: very strong evidence for FP Cox')
            elif delta > 6:
                print('   → ΔBIC > 6:  strong evidence for FP Cox')
            elif delta > 2:
                print('   → ΔBIC > 2:  positive evidence for FP Cox')
            elif abs(delta) <= 2:
                print('   → |ΔBIC| ≤ 2: models are essentially equivalent')
        except Exception:
            pass

        print(bar)
        print('Notes:')
        print('  KM     : Non-parametric; no covariates; C-index/BIC not applicable.')
        print('  Cox PH : BIC uses n_events + partial log-lik (Royston convention).')
        print('  Weibull: BIC uses n_total  + full log-lik (parametric convention).')
        print('  FP Cox : BIC uses n_events + partial log-lik; powers found by SaDE.')

    # -----------------------------------------------------------------------
    # ADD 8: Survival curve comparison plot
    # -----------------------------------------------------------------------

    def _plot_survival_curves(self, df_aft):
        """
        Plot overlaid survival curves for all four models.

        For the three covariate models (Cox PH, Weibull AFT, FP Cox) the
        prediction is made at the mean value of each covariate (the
        'average patient' profile).  KM shows the marginal curve.
        """
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))

        t_min  = self.df[self.duration_col].min()
        t_max  = self.df[self.duration_col].max()
        times  = np.linspace(t_min, t_max, 200)

        # ── Left: all-model overlay ───────────────────────────────────────────
        ax = axes[0]

        # KM marginal
        km_sf = self.km_model.survival_function_at_times(times)
        ax.plot(times, km_sf, color='gray', lw=2.5, ls='--',
                label='Kaplan-Meier (marginal)')

        # Stratified KM (if available)
        strat_colors = plt.cm.Greys(
            np.linspace(0.35, 0.75, len(self.km_strat_models)))
        for (val, kmf), sc in zip(self.km_strat_models.items(), strat_colors):
            sf = kmf.survival_function_at_times(times)
            ax.plot(times, sf, color=sc, lw=1.2, ls=':',
                    label=kmf.label)

        # Average-covariate profile for parametric/semi-parametric models
        cov_only = [c for c in self.covariates]
        mean_profile_trad = self._df_trad_final[
            [c for c in self._df_trad_final.columns
             if c not in [self.duration_col, self.event_col]]].mean()

        # Cox PH
        try:
            sf_cox = self.traditional_model.predict_survival_function(
                mean_profile_trad.to_frame().T, times=times).squeeze()
            ax.plot(times, sf_cox, color='steelblue', lw=2.5,
                    label='Cox PH (traditional, mean profile)')
        except Exception as e:
            print(f'  [!] Cox PH curve failed: {e}')

        # Weibull AFT
        if self.weibull_aft_model is not None:
            try:
                mean_profile_aft = df_aft[
                    [c for c in df_aft.columns
                     if c not in [self.duration_col, self.event_col]]].mean()
                sf_aft = self.weibull_aft_model.predict_survival_function(
                    mean_profile_aft.to_frame().T, times=times).squeeze()
                ax.plot(times, sf_aft, color='darkorange', lw=2.5, ls='-.',
                        label='Weibull AFT (mean profile)')
            except Exception as e:
                print(f'  [!] Weibull AFT curve failed: {e}')

        # FP Cox
        try:
            mean_profile_fp = self._df_fp_final[
                [c for c in self._df_fp_final.columns
                 if c not in [self.duration_col, self.event_col]]].mean()
            sf_fp = self.final_fp_model.predict_survival_function(
                mean_profile_fp.to_frame().T, times=times).squeeze()
            ax.plot(times, sf_fp, color='crimson', lw=2.5,
                    label='FP Cox (SaDE v7, mean profile)')
        except Exception as e:
            print(f'  [!] FP Cox curve failed: {e}')

        ax.set_xlabel('Time')
        ax.set_ylabel('S(t)')
        ax.set_title('Survival Curve Comparison — All 4 Models')
        ax.set_ylim(0, 1.02)
        ax.legend(fontsize=8, loc='upper right')
        ax.grid(True, alpha=0.3)

        # ── Right: KM with confidence band + log-rank note ────────────────────
        ax2 = axes[1]
        self.km_model.plot_survival_function(
            ax=ax2, ci_show=True, color='gray')
        for val, kmf in self.km_strat_models.items():
            kmf.plot_survival_function(ax=ax2, ci_show=False)

        # Add IBS annotations
        ibs_km  = self.metrics_['Kaplan-Meier']['IBS']
        ibs_cox = self.metrics_['Cox PH (trad)']['IBS']
        ibs_aft = self.metrics_['Weibull AFT']['IBS']
        ibs_fp  = self.metrics_['FP Cox (SaDE v7)']['IBS']
        anno = (f"IBS  KM={ibs_km}  Cox={ibs_cox}\n"
                f"     AFT={ibs_aft}  FP={ibs_fp}")
        ax2.text(0.02, 0.08, anno, transform=ax2.transAxes,
                 fontsize=8, va='bottom',
                 bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        ax2.set_xlabel('Time')
        ax2.set_ylabel('S(t)')
        ax2.set_title('KM Curve(s) with 95% CI')
        ax2.grid(True, alpha=0.3)

        plt.suptitle('Four-Model Survival Function Comparison',
                     fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

    # ADD 10: Model equation printer (updated for all 4 models)
    def _print_model_equations(self, _trad_unpen=None, _fp_unpen=None):
        bar = '='*72
        print(f'\n{bar}')
        print('MODEL EQUATIONS')
        print(bar)

        def _fmt_p(p):
            if p is None: return 'None'
            if p == 0:    return 'x^0 = ln(x)'
            if p == 0.5:  return 'x^0.5 = √x'
            if p == 1:    return 'x^1 (linear)'
            if p == 2:    return 'x^2 (quadratic)'
            return f'x^{p}'

        # ── 1. Kaplan-Meier ──────────────────────────────────────────────────
        print('\n── 1. KAPLAN-MEIER  (non-parametric, no covariates) ──')
        print('  S(t) = Π_{tᵢ ≤ t} (1 − dᵢ/nᵢ)')
        print('  where dᵢ = events at time tᵢ,  nᵢ = at-risk count at tᵢ')
        if self.km_model is not None:
            print(f'  Median survival time : {self.km_model.median_survival_time_:.4f}')

        # ── 2. Traditional Cox PH ────────────────────────────────────────────
        print('\n── 2. TRADITIONAL COX PH  (linear covariates) ──')
        print('  log[ h(t|x) / h₀(t) ] =')
        model_cox = _trad_unpen or self.traditional_model
        if model_cox is not None:
            for feat, coef in model_cox.params_.items():
                sign = '+' if coef >= 0 else '−'
                print(f'    {sign} {abs(coef):.6f} × {feat}')
            print(f'  k={len(model_cox.params_)},  '
                  f'C-index={self.traditional_model.concordance_index_:.4f}')

        # ── 3. Weibull AFT ───────────────────────────────────────────────────
        print('\n── 3. WEIBULL AFT  (fully parametric, log-time equation) ──')
        print('  log T = μ₀ + Σⱼ βⱼ·xⱼ + σ·ε,    ε ~ Gumbel(0, 1)')
        print('  S(t|x) = exp(−exp((log t − μ(x)) / σ))')
        print('  h(t|x) = (1/σ) · t^(1/σ − 1) · exp((log t − μ(x))/σ)')
        print()
        if self.weibull_aft_model is not None:
            params = self.weibull_aft_model.params_
            print('  lambda_ sub-model  (log-time location, μ(x)):  log T = ')
            lambda_params = params.get('lambda_', params) if hasattr(params, 'get') \
                            else params
            # lifelines stores AFT params as MultiIndex (sub-model, covariate)
            try:
                for (sub, feat), coef in params.items():
                    if sub == 'lambda_':
                        sign = '+' if coef >= 0 else '−'
                        print(f'    {sign} {abs(coef):.6f} × {feat}')
                print('  rho_ sub-model (log-shape, σ):')
                for (sub, feat), coef in params.items():
                    if sub == 'rho_':
                        print(f'    σ = exp({coef:.6f}) = {np.exp(coef):.6f}')
            except Exception:
                for feat, coef in params.items():
                    sign = '+' if coef >= 0 else '−'
                    print(f'    {sign} {abs(coef):.6f} × {feat}')
            print(f'  k={self.weibull_aft_model.params_.shape[0]},  '
                  f'C-index={self.weibull_aft_model.concordance_index_:.4f}')

        # ── 4. FP Cox ─────────────────────────────────────────────────────────
        print('\n── 4. FP COX (SaDE v7)  (FP-transformed covariates) ──')
        print('  log[ h(t|x) / h₀(t) ] =')
        model_fp = _fp_unpen or self.final_fp_model
        if model_fp is not None:
            for feat, coef in model_fp.params_.items():
                sign = '+' if coef >= 0 else '−'
                print(f'    {sign} {abs(coef):.6f} × {feat}')
            print(f'  k={len(model_fp.params_)},  '
                  f'C-index={self.final_fp_model.concordance_index_:.4f}')

        # FP power annotation
        if self.best_powers:
            print('\n  FP Power Annotation:')
            for cov, (p1, p2) in zip(self.covariates, self.best_powers):
                active  = [p for p in (p1, p2) if p is not None]
                fp_type = f'FP{len(active)}'
                scale   = self._scales.get(cov, 1.0)
                print(f'    {cov}  [{fp_type}]  (scale ÷ {scale:.4g})')
                for idx, p in enumerate(active, 1):
                    print(f'      term {idx}: {_fmt_p(p)}')
                if len(active) == 2 and active[0] == active[1]:
                    print(f'      [repeated power] term 2 = x^{active[0]} · ln(x)')
        print(bar)

    # -----------------------------------------------------------------------
    # ADD 1 (from v6): PH assumption test
    # -----------------------------------------------------------------------

    def _test_ph_assumption(self, model, df_model, model_name='Model',
                            p_threshold=0.05):
        """Grambsch-Therneau PH test via scaled Schoenfeld residuals."""
        bar = '─'*64
        print(f'\n{bar}')
        print(f'  PH ASSUMPTION TEST — {model_name}')
        print(bar)
        print('  NOTE: PH tests apply ONLY to Cox models (not KM or Weibull AFT).')
        print('  Weibull AFT does not assume PH — it uses a different framework.')
        print(f'  H₀: log-HR constant over time  |  α = {p_threshold}')

        # Formal test
        print('\n  [A] lifelines check_assumptions():')
        try:
            model.check_assumptions(df_model, p_value_threshold=p_threshold,
                                    show_plots=True, warn=False)
        except Exception as e:
            print(f'  [!] Failed: {e}')

        # Manual Pearson correlation
        print(f'\n  [B] Pearson ρ(Schoenfeld residual, ranked event time):')
        print(f'  {"-"*60}')
        any_flagged = False
        try:
            schoenfeld  = model.compute_residuals(df_model, kind='schoenfeld')
            event_mask  = df_model[self.event_col].astype(bool).values
            event_times = df_model[self.duration_col].values[event_mask]
            ranked_t    = pd.Series(event_times).rank().values
            for col in schoenfeld.columns:
                res = schoenfeld[col].values
                if len(res) != len(ranked_t): continue
                rho, pval = scipy_stats.pearsonr(res, ranked_t)
                flag = '  ⚠ |ρ|>0.2' if abs(rho) > 0.2 else ''
                if flag: any_flagged = True
                print(f'  {col:<38} ρ={rho:+.4f}  p={pval:.4f}{flag}')
            print()
            if any_flagged:
                print('  ⚠  Consider stratifying or using time-varying coefficients.')
            else:
                print('  ✓  No strong evidence of PH violation.')
        except Exception as e:
            print(f'  [!] Schoenfeld failed: {e}')
        print(bar)

    # -----------------------------------------------------------------------
    # ADD 9: Four-model cross-validation
    # -----------------------------------------------------------------------

    def run_cross_validation(self, k=5, scoring_method='concordance_index',
                             seed=None):
        """
        k-fold cross-validation for three covariate models:
            Cox PH (traditional), Weibull AFT, FP Cox.

        Kaplan-Meier is EXCLUDED from CV because:
          • It has no covariates and produces no per-subject risk score.
          • Concordance index cannot be computed.
          • Its IBS is already reported as a fixed baseline in metrics_.

        All three models are evaluated on the SAME folds.
        FP powers are FIXED (post-selection CV). Coefficients re-estimated
        per fold for all three models.

        After collecting scores, all pairwise paired t-tests are reported.
        """
        if self._df_trad_final is None or self._df_fp_final is None:
            raise RuntimeError('Call optimize() before run_cross_validation().')

        bar = '='*72
        print(f'\n{bar}')
        print(f'CROSS-VALIDATION  ({k}-fold, metric={scoring_method})')
        print(bar)
        print('  Models: Cox PH, Weibull AFT, FP Cox')
        print('  KM excluded (no covariates → no concordance index per fold).')
        print('  FP powers fixed post-selection; Cox/AFT coefficients re-estimated per fold.\n')

        if seed is not None:
            np.random.seed(seed)

        strata    = self.strata_cols or None
        fitter_kw = {'strata': strata} if strata else {}

        # --- Cox PH ---
        print('  [1/3] Cox PH...')
        scores_cox = np.array(k_fold_cross_validation(
            CoxPHFitter(penalizer=self.penalizer),
            self._df_trad_final,
            duration_col=self.duration_col, event_col=self.event_col,
            k=k, scoring_method=scoring_method, fitter_kwargs=fitter_kw,
        ))

        # --- Weibull AFT ---
        print('  [2/3] Weibull AFT...')
        scores_aft = None
        if self.weibull_aft_model is not None:
            try:
                seen2, aft_cols2 = set(), []
                for c in (self.covariates + self.strata_cols +
                          [self.duration_col, self.event_col]):
                    if c not in seen2:
                        aft_cols2.append(c)
                        seen2.add(c)
                df_aft_cv = self.df[aft_cols2].copy()
                scores_aft = np.array(k_fold_cross_validation(
                    WeibullAFTFitter(penalizer=self.penalizer),
                    df_aft_cv,
                    duration_col=self.duration_col, event_col=self.event_col,
                    k=k, scoring_method=scoring_method,
                ))
            except Exception as e:
                print(f'  [!] Weibull AFT CV failed: {e}')

        # --- FP Cox ---
        print('  [3/3] FP Cox...')
        scores_fp = np.array(k_fold_cross_validation(
            CoxPHFitter(penalizer=self.penalizer),
            self._df_fp_final,
            duration_col=self.duration_col, event_col=self.event_col,
            k=k, scoring_method=scoring_method, fitter_kwargs=fitter_kw,
        ))

        # --- Summary table ---
        W = 10
        print(f'\n  {"":35} {"Mean":>{W}} {"Std":>{W}} {"Min":>{W}} {"Max":>{W}}')
        print(f'  {"-"*67}')

        all_scores = {'Cox PH (traditional)': scores_cox,
                      'FP Cox (SaDE v7)'    : scores_fp}
        if scores_aft is not None:
            all_scores['Weibull AFT'] = scores_aft

        for name, sc in all_scores.items():
            print(f'  {name:<35} {sc.mean():>{W}.4f} {sc.std():>{W}.4f} '
                  f'{sc.min():>{W}.4f} {sc.max():>{W}.4f}')

        # --- Per-fold detail ---
        print(f'\n  Per-fold scores:')
        header = f'  {"Fold":<6}' + ''.join(f' {n:>22}' for n in all_scores)
        print(header)
        print(f'  {"-"*(6 + 22*len(all_scores))}')
        for i in range(k):
            row = f'  {i+1:<6}'
            for sc in all_scores.values():
                row += f' {sc[i]:>22.4f}'
            print(row)

        # --- Pairwise paired t-tests ---
        print(f'\n  Pairwise Paired t-tests  '
              f'(H₀: mean A = mean B, two-sided):')
        print(f'  {"-"*60}')
        score_items = list(all_scores.items())
        for (na, sa), (nb, sb) in combinations(score_items, 2):
            t_stat, p_val = scipy_stats.ttest_rel(sa, sb)
            sig = '  *' if p_val < 0.05 else ''
            better = na if sa.mean() > sb.mean() else nb
            print(f'  {na} vs {nb}')
            print(f'    t={t_stat:+.4f}  p={p_val:.4f}{sig}'
                  f'  → {better} higher mean')

        # --- Plot ---
        self._plot_cv_scores(all_scores, k, scoring_method)

        self.cv_results_ = {
            'scores'  : all_scores,
            'k'       : k,
            'scoring' : scoring_method,
        }
        return self.cv_results_

    def _plot_cv_scores(self, all_scores, k, scoring_method):
        n_models = len(all_scores)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        folds  = np.arange(1, k+1)
        colors = ['steelblue', 'darkorange', 'crimson',
                  'seagreen', 'mediumpurple'][:n_models]
        names  = list(all_scores.keys())
        metric_label = scoring_method.replace('_', ' ').title()

        # Per-fold bar chart
        ax = axes[0]
        bw  = 0.75 / n_models
        for idx, (name, sc) in enumerate(all_scores.items()):
            offset = (idx - (n_models-1)/2) * bw
            bars = ax.bar(folds + offset, sc, bw,
                          label=name, color=colors[idx], alpha=0.85)
            ax.axhline(sc.mean(), color=colors[idx], ls='--', lw=1.5, alpha=0.7)
        ax.set_xlabel('Fold')
        ax.set_ylabel(metric_label)
        ax.set_title(f'{k}-Fold CV: Score Per Fold')
        ax.set_xticks(folds)
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3, axis='y')

        # Box plot
        ax2 = axes[1]
        data = [sc for sc in all_scores.values()]
        bp   = ax2.boxplot(data, labels=names, patch_artist=True, widths=0.5)
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.6)
        for med in bp['medians']:
            med.set_color('black'); med.set_linewidth(2)
        ax2.set_ylabel(metric_label)
        ax2.set_title('CV Score Distribution')
        ax2.tick_params(axis='x', rotation=15)
        ax2.grid(True, alpha=0.3, axis='y')

        plt.suptitle(f'{k}-Fold Cross-Validation — 3 Models '
                     f'(metric: {metric_label})',
                     fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.show()

    # -----------------------------------------------------------------------
    # sklearn CV (preserved from v6)
    # -----------------------------------------------------------------------

    def run_sklearn_cv(self, k=5, seed=None):
        """scikit-learn-compatible CV via lifelines sklearn_adapter."""
        if self._df_trad_final is None:
            raise RuntimeError('Call optimize() first.')
        bar = '='*72
        print(f'\n{bar}')
        print(f'SKLEARN-COMPATIBLE CV  ({k}-fold)')
        print(bar)
        try:
            from lifelines.utils.sklearn_adapter import sklearn_adapter
            from sklearn.model_selection import cross_val_score, StratifiedKFold
        except ImportError as e:
            print(f'  [!] {e}  — falling back to lifelines native CV.')
            return self.run_cross_validation(k=k, seed=seed)
        try:
            SKCoxPH = sklearn_adapter(CoxPHFitter, event_col=self.event_col)
            drop = [self.duration_col, self.event_col] + self.strata_cols
            X_trad = self._df_trad_final.drop(
                columns=[c for c in drop if c in self._df_trad_final.columns])
            X_fp   = self._df_fp_final.drop(
                columns=[c for c in drop if c in self._df_fp_final.columns])
            y      = self._df_trad_final[self.event_col].values.astype(bool)
            cv     = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
            sc_cox = cross_val_score(
                SKCoxPH(penalizer=self.penalizer), X_trad, y, cv=cv)
            sc_fp  = cross_val_score(
                SKCoxPH(penalizer=self.penalizer), X_fp,   y, cv=cv)
            print(f'  Cox PH  (sklearn): {sc_cox.mean():.4f} ± {sc_cox.std():.4f}')
            print(f'  FP Cox  (sklearn): {sc_fp.mean():.4f} ± {sc_fp.std():.4f}')
            return {'sklearn_cox': sc_cox, 'sklearn_fp': sc_fp}
        except Exception as e:
            print(f'  [!] sklearn CV error: {e}')
            return self.run_cross_validation(k=k, seed=seed)

    # -----------------------------------------------------------------------
    # Algorithm validation (preserved from v6)
    # -----------------------------------------------------------------------

    def validate_algorithm(self, n_runs=5, seeds=None,
                           maxiter=10, popsize=6):
        """Multi-seed SaDE stability, consensus, and convergence analysis."""
        if seeds is None: seeds = list(range(n_runs))
        bar = '='*72
        print(f'\n{bar}')
        print(f'ALGORITHM VALIDATION — {n_runs} independent SaDE runs')
        print(bar)

        dim      = 2*len(self.covariates)
        pop_size = popsize*dim
        max_evals = pop_size*maxiter
        lp       = max(3, maxiter//5)
        patience = max(3, maxiter//3)

        objectives, power_selections = [], []
        convergence_hists            = []
        strategy_success_all         = []

        for run_i, seed in enumerate(seeds[:n_runs]):
            opt_tmp = FPCoxOptimizer(
                df=self.df.copy(), covariates=list(self.covariates),
                duration_col=self.duration_col, event_col=self.event_col,
                penalizer=self.penalizer, gamma=self.gamma,
                strata_cols=list(self.strata_cols))
            rng      = np.random.default_rng(seed)
            init_pop = opt_tmp._warm_start_pop(pop_size, rng)
            engine   = _SaDE(
                func=opt_tmp._objective_function,
                bounds=[(0, opt_tmp.N_POWERS-1)]*dim,
                pop_size=pop_size, max_evals=max_evals,
                lp=lp, patience=patience, seed=seed,
                callback=opt_tmp._callback)
            result   = engine.run_opt(init_pop=init_pop)

            pows = [(self.POWER_SET[result.x[2*j]],
                     self.POWER_SET[result.x[2*j+1]])
                    for j in range(len(self.covariates))]
            fp_types = ['FP1' if p[1] is None else 'FP2' for p in pows]
            print(f'  Run {run_i+1:2d} (seed={seed}): obj={result.fun:.4f}  '
                  f'gens={result.ngen}  '
                  f'powers={[(str(p[0]),str(p[1])) for p in pows]}  '
                  f'types={fp_types}')

            objectives.append(result.fun)
            power_selections.append(pows)
            convergence_hists.append(result.history)
            sr = [engine.strategy_success[i]/engine.strategy_counts[i]
                  if engine.strategy_counts[i] > 0 else 0.0
                  for i in range(_NS)]
            strategy_success_all.append(sr)

        objectives = np.array(objectives)
        cv_pct     = 100*objectives.std()/abs(objectives.mean())
        solutions  = [tuple(tuple(p) for p in run) for run in power_selections]
        most_common = max(set(solutions), key=solutions.count)
        consensus   = solutions.count(most_common)

        print(f'\n  Objective: mean={objectives.mean():.4f}  '
              f'std={objectives.std():.4f}  CV={cv_pct:.2f}%')
        print(f'  Consensus: {consensus}/{n_runs}  most_common={most_common}')
        if cv_pct < 1.0:   print('  ✓ CV<1% — excellent stability')
        elif cv_pct < 5.0: print('  ✓ CV<5% — acceptable stability')
        else:              print('  ⚠ CV≥5% — consider increasing budget')

        mean_sr = np.mean(strategy_success_all, axis=0)
        names   = ['DE/rand/1','DE/curr-to-best/2','DE/curr-to-rand/1','DE/rand/2']
        print('  Mean strategy success rates:')
        for name, sr in zip(names, mean_sr):
            print(f'    {name:<28}: {100*sr:.1f}%')

        # Plots
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        colors = plt.cm.tab10(np.linspace(0, 0.9, n_runs))

        for i, (hist, seed) in enumerate(zip(convergence_hists, seeds[:n_runs])):
            axes[0].plot(hist, alpha=0.8, color=colors[i],
                         label=f'Run {i+1} (s={seed})')
        axes[0].set(xlabel='Generation', ylabel='Best BIC',
                    title='Convergence Curves')
        axes[0].legend(fontsize=8)
        axes[0].grid(True, alpha=0.3)

        bar_c = ['steelblue' if o == objectives.min() else 'lightsteelblue'
                 for o in objectives]
        axes[1].bar([f'Run {i+1}' for i in range(n_runs)],
                    objectives, color=bar_c, edgecolor='navy', alpha=0.85)
        axes[1].axhline(objectives.mean(), color='red', ls='--', lw=2,
                        label=f'Mean={objectives.mean():.2f}')
        axes[1].fill_between([-0.5, n_runs-0.5],
                             objectives.mean()-objectives.std(),
                             objectives.mean()+objectives.std(),
                             alpha=0.15, color='red', label='±1σ')
        axes[1].set(ylabel='Best BIC', title='Objective by Run')
        axes[1].legend(fontsize=8)
        axes[1].grid(True, alpha=0.3, axis='y')
        axes[1].tick_params(axis='x', rotation=30)

        bh = axes[2].bar(np.arange(_NS), mean_sr*100, color='mediumpurple',
                         edgecolor='indigo', alpha=0.8)
        axes[2].set_xticks(np.arange(_NS))
        axes[2].set_xticklabels(
            ['DE/rand/1','curr-to-best/2','curr-to-rand/1','DE/rand/2'],
            fontsize=8, rotation=20, ha='right')
        for b, v in zip(bh, mean_sr*100):
            axes[2].text(b.get_x()+b.get_width()/2, v+0.5,
                         f'{v:.1f}%', ha='center', fontsize=9)
        axes[2].set(ylabel='Success Rate (%)', title='Strategy Success Rates')
        axes[2].grid(True, alpha=0.3, axis='y')

        plt.suptitle('SaDE Algorithm Validation', fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()

        return {'objectives': objectives, 'power_selections': power_selections,
                'consensus': f'{consensus}/{n_runs}', 'cv_pct': cv_pct}

    # -----------------------------------------------------------------------
    # Helper: FP features on arbitrary DataFrame
    # -----------------------------------------------------------------------

    def _generate_fp_features_on(self, df, features, powers):
        transformed = {}
        for col, (p1, p2) in zip(features, powers):
            x     = df[col].values.astype(float)
            log_x = np.log(x)
            active = sorted([p for p in (p1, p2) if p is not None])
            if not active: continue
            def xp(p):
                z = log_x if p == 0 else np.power(x, p)
                return z if np.isfinite(z).all() else None
            if len(active) == 1:
                p = active[0]; arr = xp(p)
                if arr is None: return None
                transformed[f'{col}_fp_{p}'] = arr
            else:
                pa, pb = active; za = xp(pa)
                if za is None: return None
                transformed[f'{col}_fp1_{pa}'] = za
                if pa == pb:
                    transformed[f'{col}_fp2_rep_{pb}'] = za * log_x
                else:
                    zb = xp(pb)
                    if zb is None: return None
                    transformed[f'{col}_fp2_{pb}'] = zb
        return transformed


print('FPCoxOptimizer v7 loaded.')

FPCoxOptimizer v7 loaded.


## Example Dataset Runs

Each `optimize()` call automatically:
1. Runs SaDE to find optimal FP powers
2. Fits **all four models**: KM, Cox PH, Weibull AFT, FP Cox
3. Prints the **four-model comparison table** with ΔBIC interpretation
4. Plots **overlaid survival curves** for all four models
5. Prints **model equations** for Cox PH, Weibull AFT, and FP Cox
6. Runs **Schoenfeld PH tests** for both Cox models

Then call:
- `run_cross_validation()` — 3-model k-fold CV with pairwise t-tests
- `validate_algorithm()` — multi-seed stability analysis

In [ ]:
import pandas as pd

# ── Dataset 1: Simulated ────────────────────────────────────────────────────
sd_df = pd.read_csv('C:/Users/HP/Desktop/research/code/data/preprocess-data/preprocess_simulated_data.csv')
optimizer_sd = FPCoxOptimizer(
    df           = sd_df,
    covariates   = ['Age'],
    duration_col = 'Time',
    event_col    = 'Event',
    strata_cols  = ['Treatment', 'Sex'],
    gamma        = 0.0,
)
optimizer_sd.optimize(maxiter=15, popsize=8, seed=42)
print('\nFP Cox model summary (Simulated):')
optimizer_sd.final_fp_model.print_summary()

cv_sd  = optimizer_sd.run_cross_validation(k=5, scoring_method='concordance_index', seed=0)
val_sd = optimizer_sd.validate_algorithm(n_runs=5, seeds=[0,1,2,3,4], maxiter=10, popsize=6)

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/RASHMIKA/Desktop/4th/research/New folder/CSC-461-8.0-Research-Project/data/preprocess-data/preprocess_simulated_data.csv'

In [ ]:
# ── Dataset 2: Haberman ─────────────────────────────────────────────────────
hm_df = pd.read_csv('C:/Users/HP/Desktop/research/code/data/preprocess-data/preprocess_haberman.csv')
optimizer_hm = FPCoxOptimizer(
    df           = hm_df,
    covariates   = ['PatientAge', 'LogNodes'],
    duration_col = 'PatientYearOperation',
    event_col    = 'SurvivalStatus',
    gamma        = 0.0,
)
optimizer_hm.optimize(maxiter=15, popsize=8, seed=42)
print('\nFP Cox model summary (Haberman):')
optimizer_hm.final_fp_model.print_summary()

cv_hm  = optimizer_hm.run_cross_validation(k=5, scoring_method='concordance_index', seed=0)
val_hm = optimizer_hm.validate_algorithm(n_runs=5, seeds=[0,1,2,3,4], maxiter=10, popsize=6)

In [ ]:
# ── Dataset 3: GBSG ─────────────────────────────────────────────────────────
gb_df = pd.read_csv('C:/Users/HP/Desktop/research/code/data/preprocess-data/preprocess_gbsg.csv')
GB_COVARIATES = ['log_pgr', 'log_nodes', 'log_er', 'age', 'size']
GB_STRATA     = ['meno', 'grade', 'hormon']
optimizer_gb = FPCoxOptimizer(
    df           = gb_df,
    covariates   = GB_COVARIATES,
    duration_col = 'rfstime',
    event_col    = 'status',
    strata_cols  = GB_STRATA,
    gamma        = 0.0,
)
optimizer_gb.optimize(maxiter=15, popsize=8, seed=42)
print('\nFP Cox model summary (GBSG):')
optimizer_gb.final_fp_model.print_summary()

cv_gb  = optimizer_gb.run_cross_validation(k=5, scoring_method='concordance_index', seed=0)
val_gb = optimizer_gb.validate_algorithm(n_runs=5, seeds=[0,1,2,3,4], maxiter=10, popsize=6)

## Interpretation Reference

### Four-model comparison guide

| | Kaplan-Meier | Cox PH (trad) | Weibull AFT | FP Cox (SaDE) |
|---|---|---|---|---|
| **Assumption** | None | PH | Weibull shape | PH |
| **Covariates** | No | Linear | Linear | FP-transformed |
| **Flexibility** | ★★★★ | ★★ | ★★ | ★★★★ |
| **Interpretability** | ★★★★ | ★★★ | ★★★ | ★★ |
| **BIC convention** | — | n_events, partial LL | n_total, full LL | n_events, partial LL |
| **Use for** | Baseline curve, log-rank test | Clinical standard | PH-free comparison | Best-fit flexible model |

### ΔBIC interpretation (Raftery 1995)

| ΔBIC | Evidence |
|---|---|
| 0 – 2 | Weak — models essentially equivalent |
| 2 – 6 | Positive — lower-BIC model preferred |
| 6 – 10 | Strong |
| > 10 | Very strong |

### Weibull AFT equation

The Weibull AFT model is parameterized as:

```
log T = μ₀ + β₁x₁ + β₂x₂ + ... + σε,  ε ~ Gumbel(0,1)
```

A positive coefficient **βⱼ** means covariate **xⱼ** *extends* survival time (accelerates the failure time in the positive direction).  This is the **opposite sign convention** from Cox PH where a positive coefficient increases the hazard.

### Why KM is excluded from CV

KM estimates the marginal survival function — it has no parameters to update, no risk score per subject, and therefore no way to compute a concordance index on a test fold. Its IBS (reported in `metrics_`) is the theoretical minimum for a model that uses no covariate information, and acts as a **sanity-check floor**: any adjusted model should beat it.

## Simulation Study

Monte-Carlo subsample simulation to compare the **stability and performance** of the three covariate models across 1 000 random draws of 90% of each dataset.

### Design
| Item | Choice | Rationale |
|---|---|---|
| Iterations | 1 000 | Sufficient to estimate means and variances reliably |
| Sample size | 90% (without replacement) | Large enough to fit models; varied enough to show variance |
| Stratification | By event status | Preserves event rate in every subsample |
| FP powers | **Fixed** (post-`optimize()`) | Avoids hours of SaDE re-runs; tests coefficient & metric stability |
| Models | Cox PH, Weibull AFT, FP Cox | KM excluded (no per-subject risk score) |

### Outputs
- **Console table**: mean ± std ± CV% ± 95% CI for every metric and coefficient
- **Plot 1**: Violin + box distributions of C-index, BIC, IBS per model
- **Plot 2**: Forest plots of coefficient stability (mean ± 95% CI)
- **`sim_results_`**: stored on the optimizer for downstream access

In [ ]:

# -----------------------------------------------------------------------
# Simulation Study — run_simulation_study()
# -----------------------------------------------------------------------
# Design rationale
# ----------------
# • FP powers are FIXED (post-selection) from the prior optimize() call.
#   Re-running SaDE 1000× would take hours; this simulation evaluates
#   coefficient and metric stability under random data resampling.
# • Each iteration draws 90% of the data WITHOUT replacement (stratified
#   by event status so the event rate is preserved per fold).
# • Three models are refitted per iteration:
#     1. Cox PH  (traditional, linear covariates)
#     2. Weibull AFT
#     3. FP Cox  (fixed powers, coefficients re-estimated)
# • Collected per iteration:
#     - All model coefficients / parameters
#     - C-index, BIC, IBS  (where applicable)
# • Summary output:
#     - Mean ± Std table for metrics and parameters
#     - Distribution plots (violin + strip) for metrics
#     - Coefficient stability plots (mean ± 95% CI)
# -----------------------------------------------------------------------

def run_simulation_study(
    self,
    n_sims: int = 1000,
    sample_frac: float = 0.90,
    seed: int = 0,
    show_progress_every: int = 100,
):
    """
    Bootstrap simulation study (subsample without replacement).

    Parameters
    ----------
    n_sims          : int   — number of simulation iterations (default 1000)
    sample_frac     : float — fraction of data sampled each iteration (default 0.90)
    seed            : int   — master RNG seed for reproducibility
    show_progress_every : int — print progress every N iterations

    Requirements
    ------------
    Must call optimize() before this method.

    Returns
    -------
    dict with keys:
        'metrics_df'   : pd.DataFrame  (n_sims × metric columns)
        'coef_cox'     : pd.DataFrame  (n_sims × Cox PH coefficient columns)
        'coef_aft'     : pd.DataFrame  (n_sims × Weibull AFT parameter columns)
        'coef_fp'      : pd.DataFrame  (n_sims × FP Cox coefficient columns)
        'summary'      : pd.DataFrame  (mean / std / CV% / 2.5% / 97.5%)
        'n_sims'       : int
        'sample_frac'  : float
    """
    if self._df_trad_final is None or self._df_fp_final is None:
        raise RuntimeError("Call optimize() before run_simulation_study().")

    bar = "=" * 72
    print(f"\n{bar}")
    print(f"SIMULATION STUDY  (n_sims={n_sims}, sample_frac={sample_frac:.0%})")
    print(bar)
    print(f"  FP powers FIXED from prior optimize() call.")
    print(f"  Models: Cox PH, Weibull AFT, FP Cox")
    print(f"  Each iteration: {int(len(self.df)*sample_frac)} / {len(self.df)} rows "
          f"(stratified by event status)\n")

    rng    = np.random.default_rng(seed)
    strata = self.strata_cols or None

    # --- Pre-build the AFT column list (same logic as _fit_final_models) ---
    seen, trad_cols = set(), []
    for c in (self.covariates + self.strata_cols +
              [self.duration_col, self.event_col]):
        if c not in seen:
            trad_cols.append(c)
            seen.add(c)

    seen2, aft_cols = set(), []
    for c in (self.covariates + self.strata_cols +
              [self.duration_col, self.event_col]):
        if c not in seen2:
            aft_cols.append(c)
            seen2.add(c)

    # --- Storage ---
    metrics_rows = []
    coef_cox_rows = []
    coef_aft_rows = []
    coef_fp_rows  = []

    n_failed = 0

    for sim_i in range(n_sims):
        # ── Stratified subsample (preserves event rate) ──────────────────
        events_idx   = self.df.index[self.df[self.event_col].astype(bool)]
        censored_idx = self.df.index[~self.df[self.event_col].astype(bool)]

        n_ev  = max(1, int(round(len(events_idx)   * sample_frac)))
        n_cen = max(1, int(round(len(censored_idx) * sample_frac)))

        sampled_ev  = rng.choice(events_idx,   size=n_ev,  replace=False)
        sampled_cen = rng.choice(censored_idx, size=n_cen, replace=False)
        idx_sample  = np.concatenate([sampled_ev, sampled_cen])

        df_sim = self.df.loc[idx_sample].copy()

        # ── Rebuild preprocessed sub-dataframes ─────────────────────────
        df_sim_trad = df_sim[trad_cols].copy()

        # FP features on the subsample
        fp_feat = self._generate_fp_features_on(
            df_sim, self.covariates, self.best_powers)
        if fp_feat is None:
            n_failed += 1
            continue

        # Build FP dataframe for the subsample
        const_cols_sim = {
            self.duration_col: df_sim[self.duration_col].values,
            self.event_col:    df_sim[self.event_col].values,
        }
        for c in self.strata_cols:
            const_cols_sim[c] = df_sim[c].values
        df_sim_fp = pd.DataFrame(const_cols_sim, index=df_sim.index)
        for col_name, arr in fp_feat.items():
            df_sim_fp[col_name] = arr

        df_sim_aft = df_sim[aft_cols].copy()

        n_ev_sim    = int(df_sim[self.event_col].sum())
        n_total_sim = len(df_sim)

        row_metrics = {}
        row_cox     = {}
        row_aft     = {}
        row_fp      = {}

        # ── 1. Cox PH ────────────────────────────────────────────────────
        try:
            cph = CoxPHFitter(penalizer=self.penalizer)
            cph.fit(df_sim_trad, duration_col=self.duration_col,
                    event_col=self.event_col, strata=strata,
                    show_progress=False)

            cph_unpen = CoxPHFitter(penalizer=0.0)
            cph_unpen.fit(df_sim_trad, duration_col=self.duration_col,
                          event_col=self.event_col, strata=strata,
                          show_progress=False)

            k_t   = len(cph_unpen.params_)
            bic_t = -2 * cph_unpen.log_likelihood_ + k_t * np.log(n_ev_sim)

            row_metrics["cox_cindex"] = cph.concordance_index_
            row_metrics["cox_bic"]    = bic_t

            # IBS (train-only for speed; consistent across iterations)
            if HAS_SKSURV:
                try:
                    y_sim = Surv.from_arrays(
                        event=df_sim[self.event_col].astype(bool).values,
                        time =df_sim[self.duration_col].values)
                    t_min = df_sim[self.duration_col].min()
                    t_max = df_sim[self.duration_col].max()
                    times = np.linspace(t_min, t_max * 0.999, 80)
                    sf_cox = cph.predict_survival_function(df_sim_trad, times=times)
                    row_metrics["cox_ibs"] = float(
                        integrated_brier_score(y_sim, y_sim, sf_cox.T.values, times))
                except Exception:
                    row_metrics["cox_ibs"] = np.nan

            for feat, coef in cph.params_.items():
                row_cox[feat] = coef

        except Exception:
            row_metrics.update({"cox_cindex": np.nan, "cox_bic": np.nan,
                                 "cox_ibs": np.nan})
            n_failed += 1

        # ── 2. Weibull AFT ───────────────────────────────────────────────
        try:
            aft = WeibullAFTFitter(penalizer=self.penalizer)
            aft.fit(df_sim_aft, duration_col=self.duration_col,
                    event_col=self.event_col, show_progress=False)

            k_w   = aft.params_.shape[0]
            ll_w  = aft.log_likelihood_
            bic_w = -2 * ll_w + k_w * np.log(n_total_sim)

            row_metrics["aft_cindex"] = aft.concordance_index_
            row_metrics["aft_bic"]    = bic_w
            row_metrics["aft_aic"]    = aft.AIC_

            if HAS_SKSURV:
                try:
                    y_sim = Surv.from_arrays(
                        event=df_sim[self.event_col].astype(bool).values,
                        time =df_sim[self.duration_col].values)
                    times = np.linspace(
                        df_sim[self.duration_col].min(),
                        df_sim[self.duration_col].max() * 0.999, 80)
                    sf_aft = aft.predict_survival_function(df_sim_aft, times=times)
                    row_metrics["aft_ibs"] = float(
                        integrated_brier_score(y_sim, y_sim, sf_aft.T.values, times))
                except Exception:
                    row_metrics["aft_ibs"] = np.nan

            # Store AFT params (MultiIndex → flatten)
            try:
                for (sub, feat), coef in aft.params_.items():
                    row_aft[f"{sub}__{feat}"] = coef
            except Exception:
                for feat, coef in aft.params_.items():
                    row_aft[str(feat)] = coef

        except Exception:
            row_metrics.update({"aft_cindex": np.nan, "aft_bic": np.nan,
                                 "aft_aic": np.nan, "aft_ibs": np.nan})

        # ── 3. FP Cox ────────────────────────────────────────────────────
        try:
            fp_m = CoxPHFitter(penalizer=self.penalizer)
            fp_m.fit(df_sim_fp, duration_col=self.duration_col,
                     event_col=self.event_col, strata=strata,
                     show_progress=False)

            fp_unpen = CoxPHFitter(penalizer=0.0)
            fp_unpen.fit(df_sim_fp, duration_col=self.duration_col,
                         event_col=self.event_col, strata=strata,
                         show_progress=False)

            k_fp   = len(fp_unpen.params_)
            bic_fp = -2 * fp_unpen.log_likelihood_ + k_fp * np.log(n_ev_sim)

            row_metrics["fp_cindex"] = fp_m.concordance_index_
            row_metrics["fp_bic"]    = bic_fp

            if HAS_SKSURV:
                try:
                    y_sim = Surv.from_arrays(
                        event=df_sim[self.event_col].astype(bool).values,
                        time =df_sim[self.duration_col].values)
                    times = np.linspace(
                        df_sim[self.duration_col].min(),
                        df_sim[self.duration_col].max() * 0.999, 80)
                    sf_fp = fp_m.predict_survival_function(df_sim_fp, times=times)
                    row_metrics["fp_ibs"] = float(
                        integrated_brier_score(y_sim, y_sim, sf_fp.T.values, times))
                except Exception:
                    row_metrics["fp_ibs"] = np.nan

            for feat, coef in fp_m.params_.items():
                row_fp[feat] = coef

        except Exception:
            row_metrics.update({"fp_cindex": np.nan, "fp_bic": np.nan,
                                 "fp_ibs": np.nan})

        # ── Store ─────────────────────────────────────────────────────────
        metrics_rows.append(row_metrics)
        coef_cox_rows.append(row_cox)
        coef_aft_rows.append(row_aft)
        coef_fp_rows.append(row_fp)

        if (sim_i + 1) % show_progress_every == 0:
            print(f"  Iteration {sim_i+1:>5} / {n_sims} complete  "
                  f"(failed so far: {n_failed})")

    # ── Assemble DataFrames ───────────────────────────────────────────────
    metrics_df  = pd.DataFrame(metrics_rows)
    coef_cox_df = pd.DataFrame(coef_cox_rows)
    coef_aft_df = pd.DataFrame(coef_aft_rows)
    coef_fp_df  = pd.DataFrame(coef_fp_rows)

    print(f"\n  Completed {n_sims} iterations  (failed: {n_failed})")

    # ── Summary statistics for metrics ───────────────────────────────────
    def _summarise(df):
        rows = []
        for col in df.columns:
            s = df[col].dropna()
            if len(s) == 0:
                continue
            rows.append({
                "metric"  : col,
                "mean"    : s.mean(),
                "std"     : s.std(),
                "cv_pct"  : 100 * s.std() / abs(s.mean()) if s.mean() != 0 else np.nan,
                "p2_5"    : s.quantile(0.025),
                "p50"     : s.median(),
                "p97_5"   : s.quantile(0.975),
                "n_valid" : int(len(s)),
            })
        return pd.DataFrame(rows).set_index("metric")

    summary_metrics = _summarise(metrics_df)

    # ── Print summary table ───────────────────────────────────────────────
    print(f"\n{bar}")
    print("SIMULATION SUMMARY — PERFORMANCE METRICS")
    print(bar)

    # Group by model
    groups = {
        "Cox PH (traditional)" : ["cox_cindex", "cox_bic", "cox_ibs"],
        "Weibull AFT"          : ["aft_cindex", "aft_bic", "aft_aic", "aft_ibs"],
        "FP Cox (SaDE)"        : ["fp_cindex",  "fp_bic",  "fp_ibs"],
    }
    metric_labels = {
        "cox_cindex": "C-index",  "cox_bic": "BIC",    "cox_ibs": "IBS",
        "aft_cindex": "C-index",  "aft_bic": "BIC",    "aft_aic": "AIC",
        "aft_ibs"   : "IBS",
        "fp_cindex" : "C-index",  "fp_bic" : "BIC",    "fp_ibs" : "IBS",
    }

    W = 12
    hdr = (f"  {'Metric':<14}{'Mean':>{W}}{'Std':>{W}}{'CV%':>{W}}"
           f"{'2.5%':>{W}}{'Median':>{W}}{'97.5%':>{W}}")

    for grp_name, cols in groups.items():
        print(f"\n  ── {grp_name} ──")
        print(hdr)
        print(f"  {'-'*78}")
        for col in cols:
            if col not in summary_metrics.index:
                continue
            r    = summary_metrics.loc[col]
            lbl  = metric_labels.get(col, col)
            print(f"  {lbl:<14}{r['mean']:>{W}.4f}{r['std']:>{W}.4f}"
                  f"{r['cv_pct']:>{W}.2f}{r['p2_5']:>{W}.4f}"
                  f"{r['p50']:>{W}.4f}{r['p97_5']:>{W}.4f}")

    # ── Parameter summary ─────────────────────────────────────────────────
    print(f"\n{bar}")
    print("SIMULATION SUMMARY — COEFFICIENTS / PARAMETERS")
    print(bar)

    for grp_name, coef_df in [
        ("Cox PH coefficients",       coef_cox_df),
        ("Weibull AFT parameters",     coef_aft_df),
        ("FP Cox coefficients",        coef_fp_df),
    ]:
        if coef_df.empty:
            continue
        summary_coef = _summarise(coef_df)
        print(f"\n  ── {grp_name} ──")
        print(hdr)
        print(f"  {'-'*78}")
        for feat in summary_coef.index:
            r = summary_coef.loc[feat]
            print(f"  {feat:<14}{r['mean']:>{W}.4f}{r['std']:>{W}.4f}"
                  f"{r['cv_pct']:>{W}.2f}{r['p2_5']:>{W}.4f}"
                  f"{r['p50']:>{W}.4f}{r['p97_5']:>{W}.4f}")

    # ── Plots ─────────────────────────────────────────────────────────────
    _plot_simulation_metrics(metrics_df, n_sims, sample_frac)
    _plot_simulation_coefficients(coef_cox_df, coef_aft_df, coef_fp_df)

    sim_results = {
        "metrics_df"  : metrics_df,
        "coef_cox"    : coef_cox_df,
        "coef_aft"    : coef_aft_df,
        "coef_fp"     : coef_fp_df,
        "summary"     : summary_metrics,
        "n_sims"      : n_sims,
        "sample_frac" : sample_frac,
    }
    self.sim_results_ = sim_results
    return sim_results


# ── Standalone plot helpers (module-level, called from the method) ───────────

def _plot_simulation_metrics(metrics_df, n_sims, sample_frac):
    """Violin + box plots for C-index, BIC, IBS across the three models."""

    metric_groups = {
        "C-index (↑ better)": {
            "Cox PH"    : "cox_cindex",
            "Weibull AFT": "aft_cindex",
            "FP Cox"    : "fp_cindex",
        },
        "BIC (↓ better)": {
            "Cox PH"    : "cox_bic",
            "Weibull AFT": "aft_bic",
            "FP Cox"    : "fp_bic",
        },
        "IBS (↓ better)": {
            "Cox PH"    : "cox_ibs",
            "Weibull AFT": "aft_ibs",
            "FP Cox"    : "fp_ibs",
        },
    }

    # Only include panels where at least one model has data
    active_groups = {
        k: v for k, v in metric_groups.items()
        if any(col in metrics_df.columns and metrics_df[col].notna().any()
               for col in v.values())
    }

    n_panels = len(active_groups)
    if n_panels == 0:
        return

    fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))
    if n_panels == 1:
        axes = [axes]

    colors = ["steelblue", "darkorange", "crimson"]

    for ax, (title, col_map) in zip(axes, active_groups.items()):
        data, labels, cols_used = [], [], []
        for lbl, col in col_map.items():
            if col in metrics_df.columns:
                d = metrics_df[col].dropna().values
                if len(d) > 0:
                    data.append(d)
                    labels.append(lbl)
                    cols_used.append(col)

        if not data:
            ax.set_visible(False)
            continue

        positions = np.arange(1, len(data) + 1)

        # Violin
        vp = ax.violinplot(data, positions=positions, showmedians=False,
                           showextrema=False)
        for pc, c in zip(vp["bodies"], colors[:len(data)]):
            pc.set_facecolor(c)
            pc.set_alpha(0.35)

        # Box inside violin
        bp = ax.boxplot(data, positions=positions, widths=0.18,
                        patch_artist=True,
                        medianprops=dict(color="black", linewidth=2),
                        whiskerprops=dict(linewidth=1.2),
                        capprops=dict(linewidth=1.2),
                        flierprops=dict(marker=".", markersize=2, alpha=0.3))
        for patch, c in zip(bp["boxes"], colors[:len(data)]):
            patch.set_facecolor(c)
            patch.set_alpha(0.65)

        # Mean marker
        for pos, d in zip(positions, data):
            ax.scatter([pos], [np.mean(d)], color="white", edgecolor="black",
                       s=45, zorder=5, label="mean" if pos == 1 else "")

        ax.set_xticks(positions)
        ax.set_xticklabels(labels, fontsize=9)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.grid(True, alpha=0.3, axis="y")
        ax.set_ylabel(title.split("(")[0].strip(), fontsize=9)
        if pos == 1:
            ax.legend(fontsize=8, loc="upper right")

    plt.suptitle(
        f"Simulation Study: Metric Distributions  "
        f"(n={n_sims}, sample={sample_frac:.0%})",
        fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


def _plot_simulation_coefficients(coef_cox_df, coef_aft_df, coef_fp_df):
    """Mean ± 95% CI forest plot for all three models' coefficients."""

    model_dfs = {
        "Cox PH"      : coef_cox_df,
        "Weibull AFT" : coef_aft_df,
        "FP Cox"      : coef_fp_df,
    }
    colors = {"Cox PH": "steelblue", "Weibull AFT": "darkorange",
              "FP Cox": "crimson"}

    active = {k: v for k, v in model_dfs.items()
              if v is not None and not v.empty}

    if not active:
        return

    n_panels = len(active)
    fig, axes = plt.subplots(1, n_panels,
                             figsize=(max(5, 4 * n_panels), 5))
    if n_panels == 1:
        axes = [axes]

    for ax, (model_name, df) in zip(axes, active.items()):
        if df.empty:
            ax.set_visible(False)
            continue

        feats  = df.columns.tolist()
        means  = df.mean()
        ci_lo  = df.quantile(0.025)
        ci_hi  = df.quantile(0.975)
        y_pos  = np.arange(len(feats))

        ax.barh(y_pos, means[feats].values,
                xerr=[means[feats].values - ci_lo[feats].values,
                      ci_hi[feats].values - means[feats].values],
                color=colors.get(model_name, "gray"), alpha=0.7,
                error_kw=dict(ecolor="black", lw=1.2, capsize=4))
        ax.axvline(0, color="black", lw=0.9, ls="--")
        ax.set_yticks(y_pos)
        ax.set_yticklabels([f[:22] for f in feats], fontsize=8)
        ax.set_xlabel("Coefficient (mean ± 95% CI)", fontsize=9)
        ax.set_title(model_name, fontsize=11, fontweight="bold")
        ax.grid(True, alpha=0.3, axis="x")

    plt.suptitle(
        "Simulation Study: Coefficient Stability  (mean ± 95% CI)",
        fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Attach to class
FPCoxOptimizer.run_simulation_study = run_simulation_study
print("run_simulation_study() attached to FPCoxOptimizer.")


In [ ]:

# ── Simulation Study ────────────────────────────────────────────────────────
#
# Run AFTER optimize() has been called on each dataset.
#
# Design
# ------
# • Each of the n_sims iterations randomly draws 90% of the dataset
#   (stratified subsample without replacement, preserving event rate).
# • FP powers are FIXED (post-selection); only coefficients are re-estimated.
# • Three models compared: Cox PH, Weibull AFT, FP Cox.
# • Collects C-index, BIC, IBS per model + all coefficients / parameters.
#
# Outputs
# -------
# • Console table: mean ± std ± CV% ± 95% CI for every metric and coefficient.
# • Plot 1: Violin + box distributions of C-index, BIC, IBS per model.
# • Plot 2: Forest plots of coefficient means with 95% CIs.

# ── Dataset 1: Simulated ─────────────────────────────────────────────────────
sim_results_sd = optimizer_sd.run_simulation_study(
    n_sims              = 1000,
    sample_frac         = 0.90,
    seed                = 42,
    show_progress_every = 100,
)

# ── Dataset 2: Haberman ──────────────────────────────────────────────────────
sim_results_hm = optimizer_hm.run_simulation_study(
    n_sims              = 1000,
    sample_frac         = 0.90,
    seed                = 42,
    show_progress_every = 100,
)

# ── Dataset 3: GBSG ──────────────────────────────────────────────────────────
sim_results_gb = optimizer_gb.run_simulation_study(
    n_sims              = 1000,
    sample_frac         = 0.90,
    seed                = 42,
    show_progress_every = 100,
)


In [ ]:

# ── Cross-dataset Simulation Comparison ──────────────────────────────────────
#
# After all three simulation studies are complete, this cell aggregates the
# mean ± std of the key metrics across datasets into a single comparison table.

def _sim_comparison_table(results_dict):
    """
    results_dict : { dataset_name : sim_results_dict }
    Prints a wide comparison table: metric × (dataset × model).
    """
    bar = "=" * 90
    print(f"\n{bar}")
    print("CROSS-DATASET SIMULATION COMPARISON — Mean (± Std)")
    print(bar)

    metric_map = {
        "C-index": [("Cox PH",      "cox_cindex"),
                    ("Weibull AFT", "aft_cindex"),
                    ("FP Cox",      "fp_cindex")],
        "BIC"    : [("Cox PH",      "cox_bic"),
                    ("Weibull AFT", "aft_bic"),
                    ("FP Cox",      "fp_bic")],
        "IBS"    : [("Cox PH",      "cox_ibs"),
                    ("Weibull AFT", "aft_ibs"),
                    ("FP Cox",      "fp_ibs")],
    }

    ds_names = list(results_dict.keys())

    for metric_name, model_cols in metric_map.items():
        print(f"\n  ── {metric_name} ──")
        # Header
        hdr = f"  {'Model':<22}"
        for ds in ds_names:
            hdr += f"  {ds:>28}"
        print(hdr)
        print(f"  {'-' * (22 + 30 * len(ds_names))}")
        for model_name, col in model_cols:
            row = f"  {model_name:<22}"
            for ds in ds_names:
                mdf = results_dict[ds]["metrics_df"]
                if col in mdf.columns:
                    s = mdf[col].dropna()
                    row += f"  {s.mean():>12.4f} ± {s.std():<12.4f}"
                else:
                    row += f"  {'N/A':>28}"
            print(row)

    print(f"\n{bar}")

_sim_comparison_table({
    "Simulated" : sim_results_sd,
    "Haberman"  : sim_results_hm,
    "GBSG"      : sim_results_gb,
})
